#  Phase 3: Baseline Model (Siamese U-Net)

In [77]:
import torch                                             
import torch.nn as nn                                 
from torchvision.models import resnet34, ResNet34_Weights
import os
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [78]:
DATA_ROOT = "/kaggle/input/datasets/azra41221/train-ert/train/train/"
LABELS_DIR = os.path.join(DATA_ROOT, "labels")
TARGETS_DIR = os.path.join(DATA_ROOT, "targets")
IMAGES_DIR = os.path.join(DATA_ROOT, "images")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class DamageDataset(Dataset):
    def __init__(self, df, images_dir, targets_dir):
        self.base_names = df["base_name"].tolist()
        self.images_dir = images_dir
        self.targets_dir = targets_dir

    def __len__(self):
        return len(self.base_names)

    def _load_image(self, path):
        image = np.array(Image.open(path))
        image = image.astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = np.transpose(image, (2, 0, 1))
        return image

    def __getitem__(self, idx):
        base_name = self.base_names[idx]

        pre = self._load_image(os.path.join(self.images_dir, f"{base_name}_pre_disaster.png"))
        post = self._load_image(os.path.join(self.images_dir, f"{base_name}_post_disaster.png"))
        mask = np.array(Image.open(os.path.join(self.targets_dir, f"{base_name}_post_disaster_target.png")))

        return torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(mask).long()


post_label_files = sorted(f for f in os.listdir(LABELS_DIR) if f.endswith("_post_disaster.json"))
sample_df = pd.DataFrame({"base_name": [f.replace("_post_disaster.json", "") for f in post_label_files[:8]]})

sample_dataset = DamageDataset(sample_df, IMAGES_DIR, TARGETS_DIR)
sample_loader = DataLoader(sample_dataset, batch_size=2, shuffle=False, num_workers=0)

print("sample size:", len(sample_dataset))

sample size: 8


## Step 1: Encoder 
pretrained ResNet34

In [79]:
class Encoder(nn.Module):

    def __init__(self):
        super().__init__()

        # load ResNet34 with its ImageNet weights
        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)
        
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  
        self.pool = resnet.maxpool          
        self.layer1 = resnet.layer1        
        self.layer2 = resnet.layer2        
        self.layer3 = resnet.layer3       
        self.layer4 = resnet.layer4 

    def forward(self, x):
        features = []

        x = self.stem(x)                   
        features.append(x)

        x = self.layer1(self.pool(x))   
        features.append(x)

        x = self.layer2(x)             
        features.append(x)

        x = self.layer3(x)            
        features.append(x)

        x = self.layer4(x)      
        features.append(x)

        return features         
                  

In [80]:
# testing
encoder = Encoder()

x = torch.randn(1, 3, 256, 256)          
features = encoder(x)                  

for i, f in enumerate(features):    
    print(f"stage {i}:", f.shape)

stage 0: torch.Size([1, 64, 128, 128])
stage 1: torch.Size([1, 64, 64, 64])
stage 2: torch.Size([1, 128, 32, 32])
stage 3: torch.Size([1, 256, 16, 16])
stage 4: torch.Size([1, 512, 8, 8])


## Step 2: Decoder

In [81]:
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.upsample = nn.Upsample(scale_factor=2)

        self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x 

In [82]:
class Decoder(nn.Module):

    def __init__(self, num_classes=5):
        super().__init__()

        self.block4 = DecoderBlock(in_channels=1024, skip_channels=512, out_channels=256)
        self.block3 = DecoderBlock(in_channels=256, skip_channels=256, out_channels=128)
        self.block2 = DecoderBlock(in_channels=128, skip_channels=128, out_channels=64)
        self.block1 = DecoderBlock(in_channels=64, skip_channels=128, out_channels=32)

        self.final_upsample = nn.Upsample(scale_factor=2)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, features):
        f0, f1, f2, f3, f4 = features

        x = self.block4(f4, f3)
        x = self.block3(x, f2)
        x = self.block2(x, f1)
        x = self.block1(x, f0)

        x = self.final_upsample(x)
        x = self.final_conv(x)
        return x

In [83]:
# test
decoder = Decoder(num_classes=5)

out = decoder(combined)
print(out.shape)

torch.Size([1, 5, 256, 256])


# Step 3: Siamese Model

In [84]:
# example: encoder
pre = torch.randn(1, 3, 256, 256)
post = torch.randn(1, 3, 256, 256)

features_pre = encoder(pre)
features_post = encoder(post)

combined = []
for f_pre, f_post in zip(features_pre, features_post):
    combined.append(torch.cat([f_pre, f_post], dim=1))

for i, f in enumerate(combined):
    print(f"stage {i}:", f.shape)

stage 0: torch.Size([1, 128, 128, 128])
stage 1: torch.Size([1, 128, 64, 64])
stage 2: torch.Size([1, 256, 32, 32])
stage 3: torch.Size([1, 512, 16, 16])
stage 4: torch.Size([1, 1024, 8, 8])


In [85]:
class SiameseUNet(nn.Module):
    def __init__(self, num_classes = 5):
        super().__init__()
        
        self.encoder = Encoder()
        self.decoder = Decoder(num_classes = num_classes)
    
    def forward(self, pre, post):

        # pre and post use same encoder 
        features_pre = self.encoder(pre)
        features_post = self.encoder(post)
        
        combined = []
        for f_pre, f_post in zip(features_pre, features_post):
            combined.append(torch.cat([f_pre, f_post], dim=1))

        return self.decoder(combined)

In [86]:
# test
model = SiameseUNet(num_classes=5)

pre = torch.randn(1, 3, 256, 256)
post = torch.randn(1, 3, 256, 256)

out = model(pre, post)
print(out.shape)

torch.Size([1, 5, 256, 256])


In [87]:
model = SiameseUNet(num_classes=5)

total_params = sum(p.numel() for p in model.parameters())
print("total parameters:", total_params)

total parameters: 26402597


### Forward pass on a real batch

In [88]:
batch = next(iter(sample_loader))

pre, post, mask = batch
print("pre:", pre.shape)
print("post:", post.shape)
print("mask:", mask.shape)

model = SiameseUNet(num_classes=5)

with torch.no_grad():
    out = model(pre, post)

print("output:", out.shape)

pre: torch.Size([2, 3, 1024, 1024])
post: torch.Size([2, 3, 1024, 1024])
mask: torch.Size([2, 1024, 1024])
output: torch.Size([2, 5, 1024, 1024])
